# Neural Network

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
# Activation functions and their derivatives
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    return a * (1 - a)  # a = sigmoid(z)

def softmax(z):
    z_shift = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def cross_entropy(y_true, y_pred, class_weights=None):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1. - eps)
    
    loss = -y_true * np.log(y_pred)
    
    if class_weights is not None:
        loss = loss * class_weights
        
    return np.mean(np.sum(loss, axis=1))

def cross_entropy_grad(y_true, y_pred, class_weights=None):
    batch_size = y_true.shape[0]
    
    if class_weights is not None:
        weighted_y_true = y_true * class_weights
        return (y_pred - weighted_y_true) / batch_size
    
    return (y_pred - y_true) / batch_size

In [4]:
class Layer:
    def __init__(self, n_in, n_out, activation='softmax'):
        limit = np.sqrt(6 / (n_in + n_out))
        self.W = np.random.uniform(-limit, limit, (n_in, n_out))
        self.b = np.zeros((1, n_out))
        self.activation = activation
        self.Z = None
        self.A = None

    def forward(self, X):
        self.Z = X @ self.W + self.b
        if self.activation == 'sigmoid':
            self.A = sigmoid(self.Z)
        elif self.activation == 'softmax':
            self.A = softmax(self.Z)
        else:
            self.A = self.Z  # linear (not used)
        return self.A

    def backward(self, dA, prev_A):
        if self.activation == 'sigmoid':
            dZ = dA * sigmoid_derivative(self.A)
        else:  
            dZ = dA
        m = prev_A.shape[0]
        dW = prev_A.T @ dZ / m
        db = np.mean(dZ, axis=0, keepdims=True)
        dPrev = dZ @ self.W.T
        # update parameters
        self.W -= self.lr * dW
        self.b -= self.lr * db
        return dPrev

    def set_lr(self, lr):
        self.lr = lr


In [ ]:
class NeuralNetwork:
    def __init__(self, layer_sizes, lr=0.01, threshold=0.5, class_weights=None):
        self.layers = []
        self.threshold = threshold
        self.class_weights = class_weights if class_weights is not None else [1.0, 1.0]
        for i in range(len(layer_sizes)-1):
            act = 'softmax' if i == len(layer_sizes)-2 else 'sigmoid'
            self.layers.append(Layer(layer_sizes[i], layer_sizes[i+1], activation=act))
        for layer in self.layers:
            layer.set_lr(lr)
        self.lr = lr

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, X, y_onehot):
        # forward pass to store activations
        self.forward(X)
        dA = cross_entropy_grad(y_onehot, self.layers[-1].A, class_weights=self.class_weights)
        # backprop through layers
        prev_A = self.layers[-2].A if len(self.layers) > 1 else X
        dPrev = self.layers[-1].backward(dA, prev_A)
        for i in range(len(self.layers)-2, -1, -1):
            cur_X = self.layers[i-1].A if i > 0 else X
            dPrev = self.layers[i].backward(dPrev, cur_X)

    def fit(self, X, y, epochs=20, batch_size=64):
        y_onehot = np.eye(2)[y]  # assumes labels are 0/1
        n_samples = X.shape[0]
        for epoch in tqdm(range(epochs), desc='NN training'):
            idx = np.random.permutation(n_samples)
            X_shuf, y_shuf = X[idx], y_onehot[idx]
            for start in range(0, n_samples, batch_size):
                end = start + batch_size
                xb, yb = X_shuf[start:end], y_shuf[start:end]
                self.backward(xb, yb)

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        y_pred_tuned = (probs >= self.threshold).astype(int)
        return y_pred_tuned

In [6]:
# Metric helpers (same as used in SVM notebook)
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)
def precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0
def recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0
def f1(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


In [174]:
# Load test data (standardized, no strings)
test_df = pd.read_csv('data/data_encoded_test.csv')
X_test = test_df.drop(columns=['is_fraud']).values
y_test = test_df['is_fraud'].values

df = pd.read_csv('data/data_encoded_train.csv')
X_train = df.drop(columns=['is_fraud']).values
y_train = df['is_fraud'].values
nn = NeuralNetwork(layer_sizes=[X_train.shape[1], 64, 32, 2], lr=0.1)
nn.fit(X_train, y_train, epochs=1000, batch_size=512)
y_pred = nn.predict(X_test)

print("Final Metrics:")
print(f'accuracy: {accuracy(y_test,y_pred)}')
print(f'recall: {recall(y_test,y_pred)}')
print(f'f1 score: {f1(y_test,y_pred)}')
print(f'precision: {precision(y_test,y_pred)}')



NN training: 100%|██████████| 1000/1000 [03:19<00:00,  5.01it/s]

Final Metrics:
accuracy: 0.6386
recall: 0.8333333333333334
f1 score: 0.01633097441480675
precision: 0.008246289169873557
